In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at23SavedTensorDefaultHooks11set_tracingEb
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:99: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. S

In [3]:
method = "GAT"
name = "GDSC1_GAT"
study = optuna.create_study(
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
df = study.trials_dataframe()
df = df.dropna(subset=[i for i in df.columns if "values" in i])
tmp = df.loc[
    df[["values_0", "values_1", "values_2", "values_3"]]
    .max(axis=1)
    .sort_values(ascending=False)
    .index
]
params = tmp[tmp["params_gnn_layer"] == method].head().iloc[0]
print(params)
params = {
    i.replace("params_", ""): j
    for i, j in zip(pd.DataFrame(params).index, params)
    if "params" in i
}

[I 2025-04-02 20:30:01,861] Using an existing study with name 'GDSC1_GAT' instead of creating a new one.


number                                            114
values_0                                     0.830902
values_1                                     0.957527
values_2                                     0.959105
values_3                                     0.803482
datetime_start             2025-03-28 17:07:49.039891
datetime_complete          2025-03-28 17:38:57.451579
duration                       0 days 00:31:08.411688
params_T_max                                      NaN
params_activation                                gelu
params_amsgrad                                  False
params_dropout1                                   0.3
params_dropout2                                   0.3
params_epochs                                     500
params_gamma_step                                 NaN
params_gnn_layer                                  GAT
params_heads                                        4
params_hidden1                                   1028
params_hidden2              

In [14]:
params

{'T_max': 0,
 'activation': 'gelu',
 'amsgrad': False,
 'dropout1': 0.3,
 'dropout2': 0.3,
 'epochs': 10,
 'gamma_step': 0,
 'gnn_layer': 'GAT',
 'heads': 4,
 'hidden1': 1028,
 'hidden2': 256,
 'hidden3': 64,
 'lr': 0.0006679884810194849,
 'momentum': 0,
 'nesterov': 0,
 'optimizer': 'Adam',
 'patience_plateau': 0,
 'scheduler': None,
 'step_size': 0,
 'thresh_plateau': 0,
 'weight_decay': 0.0013127126279820324,
 'n_drug': 331,
 'n_cell': 916,
 'n_gene': 2099}

In [4]:
data = "gdsc1"
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data(data)

load gdsc1
unique drugs: 73
unique genes: 166
DTI unique genes:  166
Top 90% variable genes:  1957
Total:  2099
Final gene exp shape: (916, 2099)
Final drug Act shape: (331, 916)


100%|██████████| 25/25 [00:06<00:00,  3.89it/s]


Done!


In [5]:
import math


def auto_convert_params(params, nan_replace=None):
    """パラメータの型を自動変換する関数

    Args:
        params (dict): 変換前のパラメータ辞書
        nan_replace: NaNの置換値（デフォルトはNone）

    Returns:
        dict: 型変換後のパラメータ辞書
    """
    converted = {}
    for k, v in params.items():
        # NaNの処理
        if isinstance(v, float) and math.isnan(v):
            converted[k] = nan_replace
        # 浮動小数点数 → 整数変換（例: 1028.0 → 1028）
        elif isinstance(v, float) and v.is_integer():
            converted[k] = int(v)
        # その他の値はそのまま保持
        else:
            converted[k] = v
    return converted

In [6]:
params = auto_convert_params(params, nan_replace=0)

In [9]:
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
    }
)

In [10]:
params

{'T_max': 0,
 'activation': 'gelu',
 'amsgrad': False,
 'dropout1': 0.3,
 'dropout2': 0.3,
 'epochs': 10,
 'gamma_step': 0,
 'gnn_layer': 'GAT',
 'heads': 4,
 'hidden1': 1028,
 'hidden2': 256,
 'hidden3': 64,
 'lr': 0.0006679884810194849,
 'momentum': 0,
 'nesterov': 0,
 'optimizer': 'Adam',
 'patience_plateau': 0,
 'scheduler': None,
 'step_size': 0,
 'thresh_plateau': 0,
 'weight_decay': 0.0013127126279820324,
 'n_drug': 331,
 'n_cell': 916,
 'n_gene': 2099}

In [11]:
PATH = f"../{data}_data/"

In [18]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for train_index, test_index in kfold.split(np.arange(pos_num)):
    sampler = RandomSampler(
        drugAct,
        train_index,
        test_index,
        null_mask,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
    )
    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )
    true_datas = pd.concat(
        [true_datas, pd.DataFrame(best_val_labels.detatch().cpu().numpy())],
        ignore_index=True,
        axis=1,
    )
    predict_datas = pd.concat(
        [predict_datas, pd.DataFrame(best_val_prob.detatch().cpu().numpy())],
        ignore_index=True,
        axis=1,
    )

true_datas.to_csv(f"true_{data}_{method}.csv")
predict_datas.to_csv(f"pred_{data}_{method}.csv")

Using device: cuda


  0%|          | 0/10 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.92 GiB. GPU 0 has a total capacity of 79.15 GiB of which 2.98 GiB is free. Process 2329129 has 1.57 GiB memory in use. Process 2329148 has 1.44 GiB memory in use. Process 2386251 has 72.07 GiB memory in use. Including non-PyTorch memory, this process has 1.07 GiB memory in use. Of the allocated memory 577.61 MiB is allocated by PyTorch, and 16.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)